In [0]:
from pyspark.sql.functions import col, avg, max, round, sum, window
from pyspark.sql.window import Window

hourly_df = spark.table("weather_project.silver.hourly_clean")

# Создаем  Daily City Weather Metrics 
daily_city_metrics = hourly_df.groupBy("city_name", "event_date").agg(
        round(avg("temperature"), 2).alias("avg_temperature"),
        round(max("wind_speed"), 2).alias("max_wind_speed"),
        round(avg("rain_probability"), 2).alias("avg_precipitation_probability")
    )

daily_city_metrics.write.mode("overwrite").saveAsTable("weather_project.gold.daily_city_metrics")

print(f"daily_city_metrics: {daily_city_metrics.count()} rows")

# Создаем Weather Trends

# оконная функция на 7 дней
window_7d = Window.partitionBy("city_name").orderBy("event_date").rowsBetween(-6, 0)

# определяем rain_frequency как процент дождливых часов в течение дня 
trends_df = hourly_df.groupBy("city_name", "event_date").agg(
        round(avg("temperature"), 2).alias("avg_temperature"),
        round(avg(col("is_rain_event").cast("int")), 2).alias("rain_frequency"))


trends_df = trends_df.withColumn("moving_avg_temperature_7d",round(avg("avg_temperature").over(window_7d), 2))

trends_df.write.mode("overwrite").saveAsTable("weather_project.gold.weather_trends")

print(f" weather_trends: {trends_df.count()} rows")

In [0]:
display(spark.table("weather_project.gold.weather_trends"))
